# LeetCode 1833 — Maximum Ice Cream Bars bằng Python

Link đề: https://leetcode.com/problems/maximum-ice-cream-bars/ · Độ khó: **Medium** · Chủ đề: Array, Greedy, Sorting, Counting Sort

Skeleton LeetCode:

```python
class Solution:
    def maxIceCream(self, costs: List[int], coins: int) -> int:
        ...
```

## 1. Mục tiêu học

1. Nhận ra bài "mua **nhiều món nhất** với ngân sách cố định" ⇒ tham lam **mua món rẻ nhất trước**, và chứng minh bằng lập luận **hoán đổi**.
2. Cài tham lam bằng **sắp xếp** (`O(n log n)`) và bằng **counting sort** (`O(n + C)`, `C = max cost`) — đúng yêu cầu "You must solve the problem by counting sort" của đề.
3. Dùng **tổng tiền tố + tìm kiếm nhị phân** (`bisect`) để trả lời "mua được bao nhiêu" và hiểu khi nào cách này có lợi (nhiều truy vấn ngân sách).
4. Viết vét cạn mọi tập con để kiểm chứng tính tối ưu của tham lam trên input nhỏ.
5. Xử lý khéo trường hợp **mua một phần** của một nhóm cùng giá trong counting sort (`min(cnt, coins // price)`).

## 2. Đề bài & ví dụ

Một ngày hè nóng, cậu bé muốn mua kem que. Cửa hàng có `n` que kem, `costs[i]` là giá que thứ `i` (đơn vị xu). Cậu bé có `coins` xu và muốn mua **nhiều que nhất có thể**. Có thể mua theo **thứ tự bất kỳ**; mỗi que mua tối đa một lần.

Trả về số que kem tối đa mua được. Đề yêu cầu giải bằng **counting sort**.

**Ví dụ 1:** `costs = [1,3,2,4,1], coins = 7` → `4` (mua các que giá `1 + 3 + 2 + 1 = 7`).

**Ví dụ 2:** `costs = [10,6,8,7,7,8], coins = 5` → `0`.

**Ví dụ 3:** `costs = [1,6,3,1,2,5], coins = 20` → `6` (tổng 18 ≤ 20 — mua hết).

**Ràng buộc:** `1 ≤ n ≤ 10⁵`; `1 ≤ costs[i] ≤ 10⁵`; `1 ≤ coins ≤ 10⁸`.

In [1]:
import random
import time
import bisect
import heapq
from itertools import accumulate, combinations
from typing import List

examples = [([1, 3, 2, 4, 1], 7, 4), ([10, 6, 8, 7, 7, 8], 5, 0), ([1, 6, 3, 1, 2, 5], 20, 6)]
for costs, coins, exp in examples:
    print(f"costs={costs}, coins={coins} -> kỳ vọng {exp}")

costs=[1, 3, 2, 4, 1], coins=7 -> kỳ vọng 4
costs=[10, 6, 8, 7, 7, 8], coins=5 -> kỳ vọng 0
costs=[1, 6, 3, 1, 2, 5], coins=20 -> kỳ vọng 6


## 3. Kiến thức nền

### 3.1 Sắp xếp: `sorted` và `list.sort`

`sorted(a)` trả list mới đã sắp, `a.sort()` sắp tại chỗ (thay đổi `a`). Cả hai `O(n log n)` (Timsort). Trong bài nộp LeetCode, sắp tại chỗ tiết kiệm bộ nhớ, nhưng trong notebook ta dùng `sorted` để không làm hỏng dữ liệu test.

In [2]:
a = [1, 3, 2, 4, 1]
b = sorted(a)
print("sorted:", b, "| a vẫn là:", a)
a.sort(); print("sau a.sort():", a)

sorted: [1, 1, 2, 3, 4] | a vẫn là: [1, 3, 2, 4, 1]
sau a.sort(): [1, 1, 2, 3, 4]


### 3.2 Counting sort (sắp xếp đếm)

Khi giá trị là số nguyên trong miền nhỏ `[1, C]`, tạo mảng `cnt` độ dài `C + 1`, đếm số lần mỗi giá trị xuất hiện; duyệt `cnt` theo chỉ số tăng dần là có thứ tự đã sắp. Thời gian `O(n + C)`, không so sánh phần tử nào.

In [3]:
costs = [1, 3, 2, 4, 1]
C = max(costs); cnt = [0] * (C + 1)
for x in costs:
    cnt[x] += 1
print("cnt =", cnt, "-> đã sắp:", [v for v in range(C + 1) for _ in range(cnt[v])])

cnt = [0, 2, 1, 1, 1] -> đã sắp: [1, 1, 2, 3, 4]


### 3.3 Tổng tiền tố và `bisect_right`

`pre[k]` = tổng `k` phần tử nhỏ nhất. Số que mua được = số `k` lớn nhất có `pre[k] ≤ coins` = `bisect_right(pre, coins) − 1` (nếu `pre[0] = 0`).

In [4]:
pre = [0] + list(accumulate(sorted([1, 3, 2, 4, 1])))
print("pre =", pre, "| coins=7 ->", bisect.bisect_right(pre, 7) - 1, "| coins=6 ->", bisect.bisect_right(pre, 6) - 1)

pre = [0, 1, 2, 4, 7, 11] | coins=7 -> 4 | coins=6 -> 3


### 3.4 Heap nhỏ nhất `heapq`

`heapify` biến list thành heap `O(n)`; `heappop` lấy phần tử nhỏ nhất `O(log n)`. Hữu ích khi chỉ cần lấy **vài** phần tử nhỏ nhất (không cần sắp toàn bộ).

In [5]:
h = [10, 6, 8, 7, 7, 8]; heapq.heapify(h)
print("3 nhỏ nhất:", [heapq.heappop(h) for _ in range(3)])

3 nhỏ nhất: [6, 7, 7]


### 3.5 Phép chia nguyên `//` để "mua tối đa bao nhiêu que cùng giá"

Với giá `p` và còn `coins` xu, số que mua tối đa (nếu có đủ hàng) là `coins // p`. Trong counting sort, nếu có `cnt[p]` que giá `p` thì mua `min(cnt[p], coins // p)`.

In [6]:
coins, p, cnt_p = 20, 6, 5
print("mua được", min(cnt_p, coins // p), "que giá", p, "| còn", coins - min(cnt_p, coins // p) * p, "xu")

mua được 3 que giá 6 | còn 2 xu


## 4. Ý tưởng nền tảng / trực giác

Muốn mua **nhiều que nhất** (không quan tâm giá trị que nào ngon hơn), mỗi que đều "giá trị" như nhau = 1. Vậy nên ưu tiên que **rẻ** để còn tiền mua thêm.

Tính tay **Ví dụ 1** (`coins = 7`): sắp giá `[1, 1, 2, 3, 4]`.

| bước | giá que | còn xu trước | mua? | còn xu sau | số que |
|---|---|---|---|---|---|
| 1 | 1 | 7 | có | 6 | 1 |
| 2 | 1 | 6 | có | 5 | 2 |
| 3 | 2 | 5 | có | 3 | 3 |
| 4 | 3 | 3 | có | 0 | 4 |
| 5 | 4 | 0 | **không** | 0 | 4 |

Đáp án 4. Lưu ý: khi gặp que **không đủ tiền**, các que sau (đắt hơn hoặc bằng) chắc chắn cũng không đủ ⇒ **dừng**.

**Câu hỏi:** liệu có khi nào bỏ một que rẻ để mua que đắt hơn lại được **nhiều que hơn**? Không — mục 10 chứng minh.

In [7]:
costs, coins, _ = examples[0]
left = coins
for c in sorted(costs):
    ok = c <= left
    print(f"giá {c}: còn {left} -> {'mua' if ok else 'dừng'}")
    if not ok:
        break
    left -= c

giá 1: còn 7 -> mua
giá 1: còn 6 -> mua
giá 2: còn 5 -> mua
giá 3: còn 3 -> mua
giá 4: còn 0 -> dừng


## 5. Cách 1 — Brute Force / baseline: thử mọi tập con

**Ý tưởng:** duyệt kích thước `k` từ lớn xuống nhỏ, thử mọi tập con `k` que; nếu có tập tổng `≤ coins` thì trả `k`. Đúng định nghĩa "mua được tối đa bao nhiêu".

In [8]:
def ice_bruteforce(costs, coins):
    n = len(costs)
    for k in range(n, 0, -1):
        if any(sum(c) <= coins for c in combinations(costs, k)):
            return k
    return 0

for costs, coins, exp in examples:
    got = ice_bruteforce(costs, coins); print(got, "kỳ vọng", exp); assert got == exp

4 kỳ vọng 4
0 kỳ vọng 0
6 kỳ vọng 6


**Độ phức tạp:** tới `2^n` tập con ⇒ chỉ dùng với `n ≤ 15`.

**Nút thắt:** ta thử mọi tổ hợp, trong khi với một `k` cố định, tập con **rẻ nhất** gồm `k` que chính là `k` que rẻ nhất — chỉ cần kiểm tra **một** tập con cho mỗi `k`!

In [9]:
costs, coins, _ = examples[0]
for k in range(1, len(costs) + 1):
    cheapest = sorted(costs)[:k]
    best_subset = min(combinations(costs, k), key=sum)
    print(f"k={k}: k que rẻ nhất {cheapest} tổng {sum(cheapest)} | tập con k phần tử có tổng nhỏ nhất {list(best_subset)} tổng {sum(best_subset)}")

k=1: k que rẻ nhất [1] tổng 1 | tập con k phần tử có tổng nhỏ nhất [1] tổng 1
k=2: k que rẻ nhất [1, 1] tổng 2 | tập con k phần tử có tổng nhỏ nhất [1, 1] tổng 2
k=3: k que rẻ nhất [1, 1, 2] tổng 4 | tập con k phần tử có tổng nhỏ nhất [1, 2, 1] tổng 4
k=4: k que rẻ nhất [1, 1, 2, 3] tổng 7 | tập con k phần tử có tổng nhỏ nhất [1, 3, 2, 1] tổng 7
k=5: k que rẻ nhất [1, 1, 2, 3, 4] tổng 11 | tập con k phần tử có tổng nhỏ nhất [1, 3, 2, 4, 1] tổng 11


## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 (tập con rẻ nhất cỡ `k` = `k` que rẻ nhất):** mua được `k` que ⇔ tổng `k` que rẻ nhất `≤ coins`. Hàm `k ↦ tổng k rẻ nhất` **tăng** ⇒ đáp án là `k` lớn nhất thỏa ⇒ duyệt giá tăng dần, mua tới khi hết tiền.

**Insight 2 (không cần so sánh):** giá chỉ trong `[1, 10⁵]` ⇒ sắp bằng **counting sort** `O(n + C)`.

**Insight 3 (mua theo nhóm):** trong counting sort, nhóm giá `p` có `cnt[p]` que; mua `t = min(cnt[p], coins // p)` que một lúc; nếu `t < cnt[p]` (không mua hết nhóm) thì **dừng** — giá sau đắt hơn, không mua được nữa.

**Quy tắc:** "nếu que rẻ nhất còn lại có giá `> coins` thì dừng; ngược lại mua nó".

**Invariant:** sau khi xét các que giá `≤ p`, đã mua đúng số que tối đa có thể trong các que giá `≤ p` với tổng ngân sách ban đầu.

### 6.1 Lập luận hoán đổi bằng ví dụ chạy được

Giả sử một cách mua **không** dùng que rẻ nhất `c_min` nhưng có mua một que `c_x > c_min`. Thay `c_x` bằng `c_min` ⇒ vẫn cùng số que, tổng tiền **giảm** `c_x − c_min ≥ 0` ⇒ vẫn hợp lệ. Lặp lại phép thay này, mọi lời giải tối ưu có thể biến thành "mua `k` que rẻ nhất". Đoạn code dưới đây thực hiện đúng phép hoán đổi trên một lời giải bất kỳ:

In [10]:
def exchange_to_cheapest(costs, chosen_idx):
    chosen = set(chosen_idx)
    order = sorted(range(len(costs)), key=lambda i: costs[i])
    steps = []
    for i in order:                                   # que rẻ nhất chưa được chọn
        if i in chosen: continue
        worst = max(chosen, key=lambda j: costs[j], default=None)
        if worst is not None and costs[worst] > costs[i]:
            chosen.remove(worst); chosen.add(i)
            steps.append(f"đổi que giá {costs[worst]} -> que giá {costs[i]}")
    return sorted(chosen), steps

costs = [5, 1, 4, 2, 3]
chosen, steps = exchange_to_cheapest(costs, [0, 2])           # một lời giải 2 que tổng 9
print("ban đầu tổng", costs[0] + costs[2], "| các bước:", steps)
print("sau hoán đổi:", [costs[i] for i in chosen], "tổng", sum(costs[i] for i in chosen))

ban đầu tổng 9 | các bước: ['đổi que giá 5 -> que giá 1', 'đổi que giá 4 -> que giá 2']
sau hoán đổi: [1, 2] tổng 3


## 7. Các cách tiếp theo

### 7.1 Cách 2 — Sắp xếp + tham lam

In [11]:
def ice_sort(costs, coins):
    cnt = 0
    for c in sorted(costs):
        if c > coins:
            break
        coins -= c
        cnt += 1
    return cnt

print([ice_sort(c, k) for c, k, _ in examples])

[4, 0, 6]


**Code nộp LeetCode (Cách 2):**

In [12]:
class SolutionSort:
    def maxIceCream(self, costs: List[int], coins: int) -> int:
        costs.sort()
        for i, c in enumerate(costs):
            if c > coins:
                return i
            coins -= c
        return len(costs)

print([SolutionSort().maxIceCream(list(c), k) for c, k, _ in examples])

[4, 0, 6]


**Độ phức tạp Cách 2:** Time `O(n log n)`, Space `O(1)` (sắp tại chỗ; Timsort dùng `O(n)` bộ đệm trong tệ nhất). Không đáp ứng yêu cầu "counting sort" của đề nhưng vẫn được chấp nhận về thời gian.

### 7.2 Cách 3 — Counting sort + mua theo nhóm (đúng yêu cầu đề)

**Giải thích từng đoạn:**
1. `cnt[p]` = số que giá `p`, với `p ∈ [1, max(costs)]`.
2. Duyệt `p` tăng dần, bỏ qua `cnt[p] = 0`.
3. `t = min(cnt[p], coins // p)`; cộng `t` vào đáp án, trừ `t·p` xu.
4. Nếu `t < cnt[p]` ⇒ không đủ tiền mua hết nhóm này ⇒ các giá sau càng không mua được ⇒ dừng.

In [13]:
def ice_counting(costs, coins):
    C = max(costs)
    cnt = [0] * (C + 1)
    for x in costs:
        cnt[x] += 1
    ans = 0
    for p in range(1, C + 1):
        if cnt[p] == 0:
            continue
        t = min(cnt[p], coins // p)
        ans += t
        coins -= t * p
        if t < cnt[p]:
            break
    return ans

print([ice_counting(c, k) for c, k, _ in examples])

[4, 0, 6]


**Code nộp LeetCode (Cách 3):** xem mục 9. **Độ phức tạp:** Time `O(n + C)` với `C = max(costs) ≤ 10⁵`, Space `O(C)`.

### 7.3 Cách 4 — Tổng tiền tố + tìm kiếm nhị phân

Sắp giá, tạo `pre[k]` = tổng `k` que rẻ nhất (dãy tăng), rồi `bisect_right(pre, coins) − 1`. Một lần tiền xử lý `O(n log n)` cho phép trả lời **nhiều truy vấn ngân sách** khác nhau trong `O(log n)` mỗi truy vấn.

In [14]:
class SolutionPrefix:
    def maxIceCream(self, costs: List[int], coins: int) -> int:
        pre = [0] + list(accumulate(sorted(costs)))
        return bisect.bisect_right(pre, coins) - 1

print([SolutionPrefix().maxIceCream(c, k) for c, k, _ in examples])
budgets = [0, 1, 2, 5, 7, 11, 100]
pre = [0] + list(accumulate(sorted(examples[0][0])))
print("nhiều ngân sách cho costs Ví dụ 1:", {b: bisect.bisect_right(pre, b) - 1 for b in budgets})

[4, 0, 6]
nhiều ngân sách cho costs Ví dụ 1: {0: 0, 1: 1, 2: 2, 5: 3, 7: 4, 11: 5, 100: 5}


**Độ phức tạp Cách 4:** Time `O(n log n)` tiền xử lý + `O(log n)` mỗi truy vấn; Space `O(n)`.

### 7.4 Cách 5 — Heap: chỉ lấy các que rẻ nhất cần thiết

`heapify` `O(n)` rồi `heappop` từng que rẻ nhất tới khi hết tiền. Tổng `O(n + k log n)` với `k` = đáp án — có lợi khi đáp án nhỏ so với `n`.

In [15]:
def ice_heap(costs, coins):
    h = list(costs); heapq.heapify(h); ans = 0
    while h and h[0] <= coins:
        coins -= heapq.heappop(h); ans += 1
    return ans

print([ice_heap(c, k) for c, k, _ in examples])

[4, 0, 6]


**Độ phức tạp Cách 5:** Time `O(n + k log n)`, Space `O(n)`.

## 8. TRACE từng bước

Trace Cách 3 (counting sort) trên **Ví dụ 3** (`costs = [1,6,3,1,2,5]`, `coins = 20`):

| p | cnt[p] | coins // p | mua t | đáp án | còn xu | dừng? |
|---|---|---|---|---|---|---|
| 1 | 2 | 20 | 2 | 2 | 18 | |
| 2 | 1 | 9 | 1 | 3 | 16 | |
| 3 | 1 | 5 | 1 | 4 | 13 | |
| 5 | 1 | 2 | 1 | 5 | 8 | |
| 6 | 1 | 1 | 1 | 6 | 2 | hết nhóm |

In [16]:
def trace_counting(costs, coins):
    C = max(costs); cnt = [0] * (C + 1)
    for x in costs: cnt[x] += 1
    print("cnt (chỉ giá có hàng):", {p: cnt[p] for p in range(C + 1) if cnt[p]})
    ans = 0
    for p in range(1, C + 1):
        if not cnt[p]: continue
        t = min(cnt[p], coins // p)
        ans += t; coins -= t * p
        stop = t < cnt[p]
        print(f"  giá {p}: có {cnt[p]}, mua {t}, đáp án {ans}, còn {coins} xu{'  -> DỪNG (không đủ tiền mua hết nhóm)' if stop else ''}")
        if stop: break
    return ans

assert trace_counting([1, 6, 3, 1, 2, 5], 20) == 6
print()
assert trace_counting([1, 3, 2, 4, 1], 7) == 4

cnt (chỉ giá có hàng): {1: 2, 2: 1, 3: 1, 5: 1, 6: 1}
  giá 1: có 2, mua 2, đáp án 2, còn 18 xu
  giá 2: có 1, mua 1, đáp án 3, còn 16 xu
  giá 3: có 1, mua 1, đáp án 4, còn 13 xu
  giá 5: có 1, mua 1, đáp án 5, còn 8 xu
  giá 6: có 1, mua 1, đáp án 6, còn 2 xu

cnt (chỉ giá có hàng): {1: 2, 2: 1, 3: 1, 4: 1}
  giá 1: có 2, mua 2, đáp án 2, còn 5 xu
  giá 2: có 1, mua 1, đáp án 3, còn 3 xu
  giá 3: có 1, mua 1, đáp án 4, còn 0 xu
  giá 4: có 1, mua 0, đáp án 4, còn 0 xu  -> DỪNG (không đủ tiền mua hết nhóm)


Trace Cách 4 (tiền tố + bisect) trên **Ví dụ 2**: `pre = [0, 6, 13, 20, 28, 36, 46]`, `coins = 5` ⇒ `bisect_right(pre, 5) = 1` ⇒ đáp án `0`.

In [17]:
pre = [0] + list(accumulate(sorted([10, 6, 8, 7, 7, 8])))
print("pre =", pre, "| bisect_right(pre, 5) =", bisect.bisect_right(pre, 5), "-> đáp án", bisect.bisect_right(pre, 5) - 1)

pre = [0, 6, 13, 20, 28, 36, 46] | bisect_right(pre, 5) = 1 -> đáp án 0


## 9. Code nộp LeetCode

Bản chọn nộp: **Cách 3** — counting sort, đúng yêu cầu đề.

In [18]:
class Solution:
    def maxIceCream(self, costs: List[int], coins: int) -> int:
        C = max(costs)
        cnt = [0] * (C + 1)
        for x in costs:
            cnt[x] += 1
        ans = 0
        for p in range(1, C + 1):
            if cnt[p] == 0:
                continue
            if p > coins:
                break
            t = min(cnt[p], coins // p)
            ans += t
            coins -= t * p
        return ans

for costs, coins, exp in examples:
    got = Solution().maxIceCream(costs, coins); print(got, exp); assert got == exp

4 4
0 0
6 6


Trong bản nộp, điều kiện dừng viết là `if p > coins: break` — tương đương với "mua không hết nhóm trước" vì sau khi mua không hết nhóm `p`, số xu còn lại `< p` nên mọi giá sau (`> p`) đều `> coins`.

## 10. Vì sao thuật toán đúng

Gọi `c_1 ≤ c_2 ≤ … ≤ c_n` là các giá đã sắp, `S_k = c_1 + … + c_k`.

**Bổ đề 1 (hoán đổi).** Trong mọi tập `T` gồm `k` que, `Σ_{T} ≥ S_k`. *Chứng minh:* sắp các giá trong `T` tăng dần `t_1 ≤ … ≤ t_k`; phần tử thứ `j` của `T` không nhỏ hơn phần tử thứ `j` của toàn mảng: `t_j ≥ c_j` (vì có ít nhất `j` phần tử `≤ t_j` trong mảng). Cộng lại ⇒ `Σ_T ≥ S_k`.

**Hệ quả.** Mua được `k` que ⇔ `S_k ≤ coins`. `S_k` tăng theo `k` ⇒ đáp án là `k` lớn nhất có `S_k ≤ coins` — chính là điều tham lam (Cách 2) và bisect (Cách 4) tính.

**Counting sort (Cách 3)** duyệt cùng thứ tự `c_1 ≤ c_2 ≤ …` theo nhóm; trong nhóm giá `p`, mua `min(cnt, coins // p)` que chính là mua lần lượt từng que cùng giá cho tới khi hết hàng hoặc hết tiền. ∎

## 11. Phân tích Time / Space Complexity

Gọi `n = len(costs) ≤ 10⁵`, `C = max(costs) ≤ 10⁵`, `k` = đáp án.

| Cách | Time | Space |
|---|---|---|
| 1. Mọi tập con | `O(2^n · n)` | `O(n)` |
| 2. Sắp + tham lam | `O(n log n)` | `O(1)` tại chỗ |
| 3. Counting sort | `O(n + C)` | `O(C)` |
| 4. Tiền tố + bisect | `O(n log n)` + `O(log n)`/truy vấn | `O(n)` |
| 5. Heap | `O(n + k log n)` | `O(n)` |

In [19]:
rng = random.Random(1833)
big = [rng.randint(1, 10 ** 5) for _ in range(10 ** 5)]
for name, f in [("sort", ice_sort), ("counting", Solution().maxIceCream), ("prefix", SolutionPrefix().maxIceCream), ("heap", ice_heap)]:
    for coins in (10 ** 6, 10 ** 8):
        t = time.perf_counter(); r = f(big, coins); dt = (time.perf_counter() - t) * 1000
        print(f"{name:<9} coins={coins:>9}: {r:>6} que ({dt:6.1f} ms)")

sort      coins=  1000000:   1420 que (  15.1 ms)
sort      coins=100000000:  14126 que (  15.4 ms)
counting  coins=  1000000:   1420 que (   3.4 ms)
counting  coins=100000000:  14126 que (   4.2 ms)
prefix    coins=  1000000:   1420 que (  19.5 ms)
prefix    coins=100000000:  14126 que (  18.3 ms)
heap      coins=  1000000:   1420 que (   3.4 ms)
heap      coins=100000000:  14126 que (  10.4 ms)


## 12. Bộ test

Đối chiếu mọi cách với vét cạn (tập con) trên mảng nhỏ ngẫu nhiên (seed cố định), thêm edge case: không mua được que nào, vừa đủ tiền mua hết, nhiều que cùng giá và chỉ mua được **một phần** nhóm, `coins` rất lớn.

In [20]:
ALL = {"sort": ice_sort, "counting": ice_counting, "SolSort": lambda c, k: SolutionSort().maxIceCream(list(c), k),
       "prefix": SolutionPrefix().maxIceCream, "heap": ice_heap, "submit": Solution().maxIceCream}
def run_all(costs, coins, with_brute=True):
    res = {k: f(list(costs), coins) for k, f in ALL.items()}
    if with_brute:
        res["brute"] = ice_bruteforce(costs, coins)
    assert len(set(res.values())) == 1, (costs, coins, res)
    return next(iter(res.values()))

for costs, coins, exp in examples:
    assert run_all(costs, coins) == exp
edge = [([5], 4, 0), ([5], 5, 1), ([2, 2, 2, 2], 7, 3), ([3, 3, 3], 9, 3), ([1] * 10, 3, 3), ([100000] * 3, 10 ** 8, 3),
        ([4, 1, 4, 1], 5, 2), ([7, 3, 3, 9], 6, 2)]
for costs, coins, exp in edge:
    assert run_all(costs, coins) == exp, (costs, coins)
rng = random.Random(18330)
for _ in range(400):
    n = rng.randint(1, 10); hi = rng.choice([3, 10, 100])
    costs = [rng.randint(1, hi) for _ in range(n)]
    run_all(costs, rng.randint(1, hi * n))
for _ in range(50):
    costs = [rng.randint(1, 10 ** 5) for _ in range(rng.randint(1, 2000))]
    run_all(costs, rng.randint(1, 10 ** 8), with_brute=False)
print("Ví dụ, edge case, 400 test có vét cạn, 50 test lớn: OK")

Ví dụ, edge case, 400 test có vét cạn, 50 test lớn: OK


### 12.1 Kiểm tra tính chất (property-based)

Ngoài đối chiếu đáp án, kiểm tra các tính chất phải đúng với **mọi** input:
1. Đáp án **không giảm** khi tăng `coins`.
2. Đáp án **không tăng** khi xóa bớt một que.
3. Tổng giá của `ans` que rẻ nhất `≤ coins`, và nếu `ans < n` thì thêm que rẻ kế tiếp sẽ vượt `coins`.

In [21]:
rng = random.Random(9)
for _ in range(300):
    costs = [rng.randint(1, 20) for _ in range(rng.randint(1, 15))]; coins = rng.randint(1, 120)
    a = Solution().maxIceCream(costs, coins)
    assert Solution().maxIceCream(costs, coins + rng.randint(0, 30)) >= a
    if len(costs) > 1:
        j = rng.randrange(len(costs))
        assert Solution().maxIceCream(costs[:j] + costs[j + 1:], coins) <= a
    srt = sorted(costs)
    assert sum(srt[:a]) <= coins and (a == len(costs) or sum(srt[:a + 1]) > coins)
print("3 tính chất đúng trên 300 input ngẫu nhiên")

3 tính chất đúng trên 300 input ngẫu nhiên


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Mua theo thứ tự ban đầu (không sắp)

In [22]:
def wrong_no_sort(costs, coins):
    cnt = 0
    for c in costs:                                   # SAI: thứ tự gốc
        if c > coins: break
        coins -= c; cnt += 1
    return cnt

costs, coins, exp = examples[0]
print("sai :", wrong_no_sort(costs, coins), "| đúng:", exp)

sai : 3 | đúng: 4


**Vì sao sai:** theo thứ tự gốc, mua `1, 3, 2` xong còn 1 xu thì gặp que giá 4 và dừng — bỏ lỡ que giá 1 ở cuối (3 que thay vì 4). **Sửa:** sắp tăng dần (đề cho phép mua theo thứ tự bất kỳ).

### Lỗi 2 — Tham lam theo thứ tự gặp: bỏ qua que quá đắt (`continue`) nhưng không sắp

In [23]:
def wrong_skip_unsorted(costs, coins):
    cnt = 0
    for c in costs:
        if c > coins: continue                        # bỏ qua que đắt, vẫn xét que sau (không sắp)
        coins -= c; cnt += 1
    return cnt

costs, coins = [3, 3, 1, 1, 1], 6                     # ví dụ tự dựng (đối chiếu lại đề gốc)
print("sai :", wrong_skip_unsorted(costs, coins), "| đúng:", Solution().maxIceCream(costs, coins), "| brute:", ice_bruteforce(costs, coins))

sai : 2 | đúng: 4 | brute: 4


**Vì sao sai:** mua ngay que giá 3 khi gặp (tham lam theo thứ tự gặp) làm tốn tiền: `3 + 3 = 6` ⇒ chỉ 2 que, trong khi mua `1 + 1 + 1 + 3 = 6` được 4 que. **Sửa:** phải sắp (hoặc counting sort) trước.

### Lỗi 3 — Counting sort nhưng mua **cả nhóm** hoặc **không mua** gì trong nhóm

In [24]:
def wrong_all_or_nothing(costs, coins):
    C = max(costs); cnt = [0] * (C + 1)
    for x in costs: cnt[x] += 1
    ans = 0
    for p in range(1, C + 1):
        if cnt[p] and cnt[p] * p <= coins:           # SAI: chỉ mua khi đủ tiền mua hết nhóm
            ans += cnt[p]; coins -= cnt[p] * p
        elif cnt[p]:
            break
    return ans

costs, coins = [2, 2, 2, 2], 7                        # ví dụ tự dựng (đối chiếu lại đề gốc)
print("sai :", wrong_all_or_nothing(costs, coins), "| đúng:", Solution().maxIceCream(costs, coins))

sai : 0 | đúng: 3


**Vì sao sai:** có thể mua **một phần** nhóm cùng giá: 7 xu mua được 3 que giá 2. **Sửa:** `t = min(cnt[p], coins // p)`.

### Lỗi 4 — Counting sort dùng `continue` sau khi mua một phần nhóm (vẫn xét giá cao hơn)

Kết quả **không sai** (vì `coins < p` nên mọi giá sau đều không mua được), nhưng lãng phí: duyệt tới tận `C = 10⁵`. Tệ hơn, nếu viết nhầm `coins // p` thành `coins // p + 1` thì sẽ âm tiền. Minh họa hiệu năng:

In [25]:
def slow_no_break(costs, coins):
    C = max(costs); cnt = [0] * (C + 1)
    for x in costs: cnt[x] += 1
    ans = 0
    for p in range(1, C + 1):
        t = min(cnt[p], coins // p); ans += t; coins -= t * p
    return ans

costs = [1] * 100 + [100000]
for f in (slow_no_break, Solution().maxIceCream):
    t0 = time.perf_counter()
    for _ in range(20): r = f(costs, 10)
    print(f"{getattr(f, '__name__', '?'):<14} -> {r} ({(time.perf_counter()-t0)*1000:.1f} ms cho 20 lần)")

slow_no_break  -> 10 (108.1 ms cho 20 lần)
maxIceCream    -> 10 (36.0 ms cho 20 lần)


**Sửa:** `break` ngay khi `p > coins`.

### Lỗi 5 — Dừng khi `c >= coins` (sai ranh giới)

In [26]:
def wrong_boundary(costs, coins):
    cnt = 0
    for c in sorted(costs):
        if c >= coins: break                          # SAI: được mua khi c == coins
        coins -= c; cnt += 1
    return cnt

costs, coins, exp = examples[0]
print("sai :", wrong_boundary(costs, coins), "| đúng:", exp)

sai : 3 | đúng: 4


**Vì sao sai:** sau khi mua `1, 1, 2` còn đúng 3 xu, que giá 3 **mua được** (`3 ≤ 3`). **Sửa:** dừng khi `c > coins`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Tối đa **số lượng** món với ngân sách" (mỗi món cùng giá trị 1) ⇒ **mua rẻ nhất trước**.
- Giá trị nguyên trong miền nhỏ (`≤ 10⁵`) ⇒ **counting sort** thay cho `sort`.
- Nhiều câu hỏi ngân sách trên cùng bộ giá ⇒ **tiền tố + bisect**.
- Nếu mỗi món có **giá trị khác nhau** (không phải 1) thì đây thành **knapsack 0/1** — tham lam không còn đúng!
- Kiểm tra ranh giới `≤` (được tiêu hết tiền).

## 15. Biến thể của bài

1. **Mỗi que có giá trị `v_i`, tối đa tổng giá trị** ⇒ knapsack 0/1 (DP `O(n · coins)`), tham lam sai (Bài tập 5 minh họa).
2. **Nhiều truy vấn ngân sách** ⇒ Cách 4.
3. **Mỗi loại que có số lượng `q_i`** (đã ở dạng nhóm) ⇒ chính là vòng lặp counting sort.
4. **1798 Maximum Number of Consecutive Values You Can Make**, **2279 Maximum Bags With Full Capacity of Rocks**, **1196 How Many Apples Can You Put into the Basket**: tham lam "nhỏ nhất trước".
5. **Phải mua ít nhất một que mỗi loại hương vị**: thêm ràng buộc nhóm — bài khó hơn.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Tập con | Thử mọi tổ hợp | `O(2^n n)` | `O(n)` | Kiểm chứng |
| 2. Sắp + tham lam | Rẻ nhất trước | `O(n log n)` | `O(1)` | Code ngắn nhất |
| 3. Counting sort | Đếm theo giá, mua theo nhóm | `O(n + C)` | `O(C)` | **Bản nộp** (yêu cầu đề) |
| 4. Tiền tố + bisect | `S_k ≤ coins` | `O(n log n)` | `O(n)` | Nhiều truy vấn ngân sách |
| 5. Heap | Lấy rẻ nhất dần | `O(n + k log n)` | `O(n)` | Đáp án nhỏ so với `n` |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** (a) `([1, 1, 1], 2)`; (b) `([5, 4, 3, 2, 1], 15)`; (c) `([9, 9, 9], 26)`; (d) `([2, 5, 3, 7], 9)`.

**Bài 2 (tính tay):** Với `costs = [4, 2, 2, 6, 1, 2]`, `coins = 9`: lập bảng counting sort (giá, số lượng, mua, còn xu).

**Bài 3 (điền TODO):** Hoàn thiện `counts_of(costs)` trả mảng đếm `cnt` độ dài `max(costs) + 1`.

**Bài 4 (tự viết lại):** Viết `which_bars(costs, coins)` trả **danh sách chỉ số** các que mua (tối đa số que; khi hòa giá chọn chỉ số nhỏ hơn trước) và kiểm tra tổng giá `≤ coins`.

**Bài 5 (mini challenge):** Mỗi que có thêm "độ ngon" `taste[i]`. Viết `max_taste(costs, taste, coins)` tối đa **tổng độ ngon** (knapsack 0/1, `coins ≤ 1000`), và tìm một ví dụ nhỏ cho thấy tham lam "rẻ nhất trước" **không** tối ưu cho bài này.

In [27]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def counts_of_todo(costs):
    cnt = [0] * (max(costs) + 1)
    # TODO: với mỗi x trong costs, tăng cnt[x]
    return cnt

In [28]:
# Bài 4, 5 — khung tự viết (chỉ định nghĩa, không gọi)
def which_bars(costs, coins):
    # TODO: sắp chỉ số theo (giá, chỉ số); mua lần lượt khi còn đủ tiền
    pass

def max_taste(costs, taste, coins):
    # TODO: dp[c] = tổng độ ngon lớn nhất với ngân sách c; duyệt c giảm dần cho mỗi que (0/1)
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** Mức 1: sắp. Mức 2: cộng dồn. Mức 3: (a) `2`; (b) `5` (tổng đúng 15); (c) `2`; (d) sắp `[2, 3, 5, 7]`: `2 + 3 = 5 ≤ 9`, thêm `5` thành `10 > 9` ⇒ `2`.

**Bài 2.** Mức 1: `cnt`: giá 1 ×1, giá 2 ×3, giá 4 ×1, giá 6 ×1. Mức 2: mua 1 que giá 1 (còn 8), 3 que giá 2 (còn 2). Mức 3: giá 4 > 2 ⇒ dừng; đáp án `4`.

**Bài 3.** Mức 1: mảng `max + 1` phần tử. Mức 2: `cnt[x] += 1`. Mức 3: tổng `cnt` phải bằng `n`.

**Bài 4.** Mức 1: `order = sorted(range(n), key=lambda i: (costs[i], i))`. Mức 2: mua tới khi giá `> coins`. Mức 3: số que bằng `Solution().maxIceCream`.

**Bài 5.** Mức 1: DP knapsack 0/1. Mức 2: `for c in range(coins, cost − 1, −1): dp[c] = max(dp[c], dp[c − cost] + taste)`. Mức 3: phản ví dụ: `costs = [1, 1, 2]`, `taste = [1, 1, 10]`, `coins = 2` — tham lam rẻ nhất lấy hai que giá 1 (ngon 2), tối ưu là que giá 2 (ngon 10).

## 19. Đáp án tham khảo cho bài tập

In [29]:
# Bài 1, 2, 3
b1 = [Solution().maxIceCream(c, k) for c, k in [([1, 1, 1], 2), ([5, 4, 3, 2, 1], 15), ([9, 9, 9], 26), ([2, 5, 3, 7], 9)]]
print("Bài 1:", b1); assert b1 == [2, 5, 2, 2]
print("Bài 2:"); assert trace_counting([4, 2, 2, 6, 1, 2], 9) == 4
def counts_of_todo(costs):
    cnt = [0] * (max(costs) + 1)
    for x in costs:
        cnt[x] += 1
    return cnt
assert counts_of_todo([1, 3, 2, 4, 1]) == [0, 2, 1, 1, 1] and sum(counts_of_todo([5, 5, 2])) == 3
print("Bài 3: OK", counts_of_todo([1, 3, 2, 4, 1]))

Bài 1: [2, 5, 2, 2]
Bài 2:
cnt (chỉ giá có hàng): {1: 1, 2: 3, 4: 1, 6: 1}
  giá 1: có 1, mua 1, đáp án 1, còn 8 xu
  giá 2: có 3, mua 3, đáp án 4, còn 2 xu
  giá 4: có 1, mua 0, đáp án 4, còn 2 xu  -> DỪNG (không đủ tiền mua hết nhóm)
Bài 3: OK [0, 2, 1, 1, 1]


In [30]:
# Bài 4, 5
def which_bars(costs, coins):
    chosen = []
    for i in sorted(range(len(costs)), key=lambda i: (costs[i], i)):
        if costs[i] > coins:
            break
        coins -= costs[i]; chosen.append(i)
    return chosen

def max_taste(costs, taste, coins):
    dp = [0] * (coins + 1)
    for c, t in zip(costs, taste):
        for b in range(coins, c - 1, -1):
            dp[b] = max(dp[b], dp[b - c] + t)
    return dp[coins]

def greedy_taste(costs, taste, coins):
    return sum(taste[i] for i in which_bars(costs, coins))

rng = random.Random(4)
for _ in range(300):
    costs = [rng.randint(1, 9) for _ in range(rng.randint(1, 8))]; coins = rng.randint(1, 30)
    ch = which_bars(costs, coins)
    assert len(ch) == Solution().maxIceCream(costs, coins) and sum(costs[i] for i in ch) <= coins
    taste = [rng.randint(1, 10) for _ in costs]
    brute = max((sum(taste[i] for i in s) for k in range(len(costs) + 1) for s in combinations(range(len(costs)), k)
                 if sum(costs[i] for i in s) <= coins), default=0)
    assert max_taste(costs, taste, coins) == brute
print("Bài 4:", which_bars([1, 3, 2, 4, 1], 7))
print("Bài 5: phản ví dụ tham lam:", greedy_taste([1, 1, 2], [1, 1, 10], 2), "<", max_taste([1, 1, 2], [1, 1, 10], 2))

Bài 4: [0, 4, 2, 1]
Bài 5: phản ví dụ tham lam: 2 < 10


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template counting sort + mua theo nhóm:**

```python
cnt = [0] * (max(costs) + 1)
for x in costs: cnt[x] += 1
ans = 0
for p in range(1, len(cnt)):
    if cnt[p] == 0: continue
    if p > coins: break
    t = min(cnt[p], coins // p)
    ans += t; coins -= t * p
return ans
```

**Checklist:**
- [ ] Mua **rẻ nhất trước** (sắp hoặc đếm), không theo thứ tự gốc?
- [ ] Ranh giới `p ≤ coins` (được tiêu hết tiền)?
- [ ] Counting sort: mua **một phần** nhóm `min(cnt[p], coins // p)`?
- [ ] Dừng sớm khi `p > coins`?
- [ ] Đã test: không mua được que nào, đủ tiền mua hết, nhóm cùng giá mua một phần, `coins = 10⁸`?